# Hey Datathon 2026 — 03 · Preprocessing & Feature Engineering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mmateo101/hey-datathon-2026/blob/main/notebooks/03_preprocessing_features.ipynb)

> **Propósito**: Auditoría completa del pipeline de datos para revisión del equipo.
> Cada transformación muestra el estado **antes** y **después**.
> Cada decisión está justificada en la celda markdown que la precede.

| Sección | Contenido |
|---|---|
| 1 | Carga de datos raw |
| 2 | Auditoría de nulos y estrategia de imputación |
| 3 | Verificación y corrección de tipos |
| 4 | Análisis de distribuciones y outliers |
| 5 | Cálculo de gaps entre transacciones |
| 6 | Construcción de etiquetas healthy / recovered / churned |
| 7 | Feature engineering RFM |
| 8 | Features de perfil del cliente |
| 9 | Dataset final para modelado |
| 10 | Guardar |
| 11 | Resumen ejecutivo de decisiones |

In [ ]:
# ── Celda 0: Setup ────────────────────────────────────────────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/hey-datathon-2026'
    sys.path.insert(0, BASE)
    DATA_PATH      = BASE + '/data/raw/'
    PROCESSED_PATH = BASE + '/data/processed/'
    !pip install -q pandas numpy matplotlib seaborn
else:
    BASE = os.path.abspath('..')
    sys.path.insert(0, BASE)
    DATA_PATH      = '../data/raw/'
    PROCESSED_PATH = '../data/processed/'

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

from src.preprocessing import load_data, merge_clientes_etiquetados
from src.features import (
    calcular_gaps, etiquetar_clientes,
    build_rfm_features, build_model_features,
    FEATURES_PERFIL, GAP_CHURNED,
    GAP_RECOVERED_MIN, GAP_RECOVERED_MAX, VENTANA_ACTIVO_DIAS
)

# ── Paleta Hey Banco ──────────────────────────────────────────────────────────
HEY_GREEN  = '#00C48C'
HEY_DARK   = '#2D3142'
HEY_ORANGE = '#FFB347'
HEY_COLORS = {'healthy': HEY_GREEN, 'recovered': HEY_ORANGE, 'churned': HEY_DARK}

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110,
                     'axes.spines.top': False,
                     'axes.spines.right': False})

print('Entorno      : ' + ('Google Colab' if IN_COLAB else 'local'))
print('DATA_PATH    : ' + DATA_PATH)
print('PROCESSED    : ' + PROCESSED_PATH)

---
## 1. Carga de datos raw

### ¿Qué estamos cargando y por qué estos tres archivos?

| Archivo | Rol en el modelo | Por qué lo necesitamos |
|---|---|---|
| `hey_clientes.csv` | Features de perfil + target indirecto | Contiene variables demográficas, de producto y de satisfacción que predicen churn |
| `hey_transacciones.csv` | Fuente de features RFM + definición de etiquetas | El comportamiento transaccional es la señal más fuerte de desenganche |
| `hey_productos.csv` | Referencia (no entra al modelo directamente) | Útil para enriquecer el análisis de la transacción de regreso |

**`load_data()` aplica automáticamente:**
- Parseo de fecha (`fecha_hora` → `fecha` como `datetime64`)
- Normalización de booleanos en clientes (`True`/`False` en lugar de `0`/`1`)

In [ ]:
# ── Celda 1: Carga de datos raw ───────────────────────────────────────────────
clientes, productos, transacciones = load_data(DATA_PATH)

print('═══ SHAPES ═══')
print(f'hey_clientes.csv      : {clientes.shape[0]:>6,} filas × {clientes.shape[1]} cols')
print(f'hey_productos.csv     : {productos.shape[0]:>6,} filas × {productos.shape[1]} cols')
print(f'hey_transacciones.csv : {transacciones.shape[0]:>6,} filas × {transacciones.shape[1]} cols')

# .info() resumido
for nombre, df in [('CLIENTES', clientes), ('TRANSACCIONES', transacciones)]:
    print(f'\n─── {nombre} — .info() ───')
    df.info()

# Primeras filas
print('\n─── CLIENTES — primeras 5 filas ───')
display(clientes.head())

print('\n─── TRANSACCIONES — primeras 5 filas ───')
display(transacciones.head())

---
## 2. Auditoría de nulos

### ¿Qué columnas tienen nulos y cuál es nuestra estrategia?

| Columna | Dataset | Nulos esperados | Estrategia | Justificación |
|---|---|---|---|---|
| `satisfaccion_1_10` | clientes | ~751 (5%) | **Imputar con mediana por etiqueta** (en el modelo) | La encuesta no siempre se completa; la mediana preserva la distribución por grupo |
| `estado` | clientes | ~432 (2.9%) | **Dejar como NaN / excluir del modelo** | No entra al modelo; no vale la pena imputar geolocalización |
| `ciudad` | clientes | ~432 (2.9%) | **Dejar como NaN / excluir del modelo** | Misma razón que `estado` |
| `gap_dias` | transacciones (calculado) | Primera TX de cada cliente | **Es esperado — no imputar** | La primera transacción no tiene anterior; el NaN es correcto |

> **Regla general**: si una columna tiene > 30% de nulos y no es crítica, se descarta.
> Si es crítica (como `satisfaccion_1_10`), se imputa con la mediana del grupo correspondiente.

In [ ]:
# ── Celda 2: Auditoría de nulos ───────────────────────────────────────────────

def tabla_nulos(df, nombre):
    n   = df.isnull().sum()
    pct = (n / len(df) * 100).round(2)
    tabla = pd.DataFrame({'n_nulos': n, 'pct_nulos': pct})
    tabla = tabla[tabla['n_nulos'] > 0].sort_values('pct_nulos', ascending=False)
    print(f'\n── {nombre} ({len(df):,} filas) ──')
    if tabla.empty:
        print('  Sin nulos')
    else:
        display(tabla)
    return tabla

nulos_cli = tabla_nulos(clientes,      'hey_clientes.csv')
nulos_prd = tabla_nulos(productos,     'hey_productos.csv')
nulos_tx  = tabla_nulos(transacciones, 'hey_transacciones.csv')

# Visualizar nulos de clientes
if not nulos_cli.empty:
    fig, ax = plt.subplots(figsize=(8, 3))
    nulos_cli.sort_values('pct_nulos').plot.barh(
        y='pct_nulos', ax=ax, color=HEY_ORANGE, legend=False
    )
    ax.set_title('% de nulos por columna — hey_clientes.csv', fontweight='bold')
    ax.set_xlabel('% nulos')
    ax.axvline(30, color=HEY_DARK, linestyle='--', alpha=0.6, label='Umbral 30%')
    ax.legend()
    plt.tight_layout()
    plt.show()

# Diagnóstico
print('\n─── DIAGNÓSTICO ───')
print(f'satisfaccion_1_10 : {clientes["satisfaccion_1_10"].isnull().sum():,} nulos '
      f'({clientes["satisfaccion_1_10"].isnull().mean()*100:.1f}%) → imputar mediana en modelo')
print(f'estado / ciudad   : {clientes["estado"].isnull().sum():,} nulos '
      f'({clientes["estado"].isnull().mean()*100:.1f}%) → excluir del modelo')

---
## 3. Limpieza de tipos de datos

### ¿Qué corregimos y por qué?

| Columna | Tipo raw (CSV) | Tipo corregido | Motivo |
|---|---|---|---|
| `fecha_hora` | `object` | `datetime64[ns]` → renombrado a `fecha` | Necesario para calcular gaps y recencia |
| `es_hey_pro` | `object` ("True"/"False") | `bool` | Evita que pandas lo trate como categoría de alta cardinalidad |
| `nomina_domiciliada` | `object` | `bool` | Ídem |
| `recibe_remesas` | `object` | `bool` | Ídem |
| `usa_hey_shop` | `object` | `bool` | Ídem |
| `tiene_seguro` | `object` | `bool` | Ídem |
| `patron_uso_atipico` | `object` | `bool` | Ídem |

> Todas estas transformaciones las aplica `load_data()` automáticamente.
> Esta celda sirve como **verificación auditora** de que el resultado es el esperado.

In [ ]:
# ── Celda 3: Verificación de tipos ────────────────────────────────────────────

# Leer CSV crudo para mostrar el ANTES (sin load_data)
_raw_cli = pd.read_csv(DATA_PATH + 'hey_clientes.csv')
_raw_tx  = pd.read_csv(DATA_PATH + 'hey_transacciones.csv')

BOOL_COLS = ['es_hey_pro', 'nomina_domiciliada', 'recibe_remesas',
             'usa_hey_shop', 'tiene_seguro', 'patron_uso_atipico']

# Tabla comparativa ANTES / DESPUÉS
comparativa = []
for col in BOOL_COLS:
    if col in _raw_cli.columns and col in clientes.columns:
        comparativa.append({
            'columna':    col,
            'dtype_raw':  str(_raw_cli[col].dtype),
            'dtype_proc': str(clientes[col].dtype),
            'OK':         clientes[col].dtype == bool,
        })

# Fecha
fecha_col_raw = 'fecha_hora' if 'fecha_hora' in _raw_tx.columns else 'fecha'
comparativa.append({
    'columna':    fecha_col_raw + ' → fecha',
    'dtype_raw':  str(_raw_tx[fecha_col_raw].dtype),
    'dtype_proc': str(transacciones['fecha'].dtype),
    'OK':         'datetime' in str(transacciones['fecha'].dtype),
})

df_comp = pd.DataFrame(comparativa)
print('═══ COMPARATIVA DE TIPOS ═══')
display(df_comp)

# Verificación final
todo_ok = df_comp['OK'].all()
status  = 'TODOS LOS TIPOS CORRECTOS' if todo_ok else 'ADVERTENCIA: revisar tipos incorrectos'
print(f'\nResultado: {status}')

# Rango de fechas
print(f'\nRango temporal de transacciones:')
print(f'  Desde : {transacciones["fecha"].min().date()}')
print(f'  Hasta : {transacciones["fecha"].max().date()}')
print(f'  Span  : {(transacciones["fecha"].max() - transacciones["fecha"].min()).days} días')

---
## 4. Análisis de distribuciones y outliers

### ¿Hay outliers que debamos tratar?

**Decisiones tomadas:**

| Variable | Outliers detectados | Decisión | Justificación |
|---|---|---|---|
| `ingreso_mensual_mxn` | Algunos valores > $80k MXN | **Conservar** | Representan segmento premium real; el modelo XGBoost es robusto a outliers |
| `score_buro` | Rango 295-850, sin outliers | **Sin acción** | Escala definida por Buró de Crédito |
| `edad` | Rango 18-60, sin outliers | **Sin acción** | Constrainado por negocio (Hey Banco atiende adultos) |
| `antiguedad_dias` | Máximo ~1825 días (~5 años) | **Sin acción** | Rango coherente con la historia del banco |
| `monto` (transacciones) | Cola larga hacia valores altos | **Cap en p95 para visualización** | No se capea para el modelo; solo para claridad en los gráficos |

> XGBoost usa árboles de decisión que son **invariantes a monotransformaciones** de features.
> Por tanto, no es necesario normalizar ni capear para el modelo.
> Si se usara regresión logística, se requeriría StandardScaler.

In [ ]:
# ── Celda 4: Distribuciones y outliers ───────────────────────────────────────

NUM_VARS_CLI = ['ingreso_mensual_mxn', 'score_buro', 'edad', 'antiguedad_dias']

fig, axes = plt.subplots(2, 3, figsize=(18, 9))
axes = axes.flatten()

for i, var in enumerate(NUM_VARS_CLI):
    data  = clientes[var].dropna()
    media = data.mean()
    med   = data.median()
    p95   = data.quantile(0.95)

    axes[i].hist(data, bins=40, color=HEY_GREEN, edgecolor='white', alpha=0.85)
    axes[i].axvline(media, color=HEY_DARK,   lw=2, linestyle='--',
                    label=f'Media:   {media:,.0f}')
    axes[i].axvline(med,   color=HEY_ORANGE, lw=2, linestyle='-',
                    label=f'Mediana: {med:,.0f}')
    axes[i].axvline(p95,   color='red',      lw=1, linestyle=':',
                    label=f'p95:     {p95:,.0f}')
    axes[i].set_title(var.replace('_', ' ').title(), fontweight='bold')
    axes[i].legend(fontsize=8)

# Monto de transacciones (cap en p95 para claridad)
monto_data = transacciones['monto'].dropna()
p95_monto  = monto_data.quantile(0.95)
axes[4].hist(monto_data.clip(0, p95_monto), bins=50,
             color=HEY_ORANGE, edgecolor='white', alpha=0.85)
axes[4].set_title(f'Monto transacciones (capeado en p95 = ${p95_monto:,.0f})',
                  fontweight='bold')
axes[4].set_xlabel('MXN')

axes[5].set_visible(False)

plt.suptitle('Distribuciones de variables clave — pre-limpieza',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Estadísticas descriptivas
print('═══ ESTADÍSTICAS DESCRIPTIVAS — CLIENTES ═══')
display(clientes[NUM_VARS_CLI].describe().round(2))

print('\n═══ ESTADÍSTICAS DESCRIPTIVAS — MONTO (transacciones) ═══')
print(monto_data.describe().round(2).to_string())
print(f'p95  : ${p95_monto:,.2f}')
print(f'p99  : ${monto_data.quantile(0.99):,.2f}')
print(f'max  : ${monto_data.max():,.2f}')

---
## 5. Cálculo de gaps entre transacciones

### ¿Cómo definimos inactividad? Justificación de los umbrales

Un **gap** es el número de días entre dos transacciones consecutivas de un mismo cliente.
Es la materia prima para definir quién está en riesgo.

**¿Por qué 45 días como inicio de riesgo?**
- El cliente promedio activo transacciona cada ~7-15 días (frecuencia semanal/quincenal)
- Un silencio de 45 días (~1.5 meses) representa una desviación de 3-6 ciclos normales
- Por debajo de 45 días, los gaps son ruido estacional (vacaciones, viajes)

**¿Por qué 90 días como umbral de churn?**
- 90 días (~3 meses) sin transaccionar es el umbral estándar en banca digital
- A partir de este punto, el costo de reactivación supera el beneficio esperado
- Alineado con los criterios regulatorios de cuentas inactivas en México (CNBV)

**¿Por qué 30 días de ventana activa para `healthy`?**
- Un cliente que transaccionó en el último mes es definitivamente activo
- Esta ventana cubre al menos un ciclo de nómina

In [ ]:
# ── Celda 5: Cálculo de gaps ──────────────────────────────────────────────────

tx_con_gaps = calcular_gaps(transacciones)

gaps_validos   = tx_con_gaps['gap_dias'].dropna()
gaps_por_cli   = tx_con_gaps.groupby('user_id')['gap_dias'].max().dropna()

print('═══ RESUMEN DE GAPS ═══')
print(f'Total transacciones       : {len(tx_con_gaps):,}')
print(f'Gaps calculados           : {len(gaps_validos):,}  (NaN en primera TX de cada cliente)')
print(f'Clientes con > 1 TX       : {gaps_por_cli.count():,}')

print('\n─── Estadísticas de gap_dias ───')
print(gaps_validos.describe().round(1).to_string())

print('\n─── Percentiles clave ───')
for p in [50, 75, 90, 95, 99]:
    val = np.percentile(gaps_validos, p)
    marker = ' ← umbral churn' if p == 90 else ''
    print(f'  p{p:>2} : {val:>6.0f} días{marker}')

# Clientes en cada zona de riesgo
en_zona_riesgo = ((gaps_por_cli >= GAP_RECOVERED_MIN) &
                  (gaps_por_cli <= GAP_RECOVERED_MAX)).sum()
sobre_churn    = (gaps_por_cli >= GAP_CHURNED).sum()

print(f'\n─── Clientes por zona de gap máximo ───')
print(f'  gap < {GAP_RECOVERED_MIN}d (sano)          : {(gaps_por_cli < GAP_RECOVERED_MIN).sum():,}')
print(f'  gap {GAP_RECOVERED_MIN}-{GAP_RECOVERED_MAX}d (zona de riesgo): {en_zona_riesgo:,}')
print(f'  gap >= {GAP_CHURNED}d (posible churn): {sobre_churn:,}')

# ── Gráficas ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histograma de todos los gaps (hasta 180d)
axes[0].hist(gaps_validos[gaps_validos <= 180], bins=60,
             color=HEY_GREEN, edgecolor='white', alpha=0.85)
axes[0].axvspan(GAP_RECOVERED_MIN, GAP_RECOVERED_MAX,
                alpha=0.15, color=HEY_ORANGE, label=f'Zona riesgo ({GAP_RECOVERED_MIN}-{GAP_RECOVERED_MAX}d)')
axes[0].axvline(GAP_CHURNED, color=HEY_DARK, lw=2, linestyle='--',
                label=f'Umbral churn ({GAP_CHURNED}d)')
axes[0].set_title('Distribución de gaps entre TXs (hasta 180d)', fontweight='bold')
axes[0].set_xlabel('Días entre transacciones')
axes[0].set_ylabel('Transacciones')
axes[0].legend()

# Gap máximo por cliente
axes[1].hist(gaps_por_cli[gaps_por_cli <= 200], bins=50,
             color=HEY_ORANGE, edgecolor='white', alpha=0.85)
axes[1].axvspan(GAP_RECOVERED_MIN, GAP_RECOVERED_MAX,
                alpha=0.15, color=HEY_ORANGE, label='Zona recovered')
axes[1].axvline(GAP_CHURNED, color=HEY_DARK, lw=2, linestyle='--',
                label=f'Umbral churn ({GAP_CHURNED}d)')
axes[1].set_title('Gap máximo histórico por cliente (hasta 200d)', fontweight='bold')
axes[1].set_xlabel('Días (gap máximo del cliente)')
axes[1].set_ylabel('Clientes')
axes[1].legend()

plt.suptitle('Análisis de inactividad entre transacciones', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. Construcción de etiquetas

### Proceso de etiquetado: `healthy` / `recovered` / `churned`

**Algoritmo de prioridad** (aplicado por `etiquetar_clientes()`):

```
SI el cliente tuvo gap 45-90d Y tuvo TXs después:
    → recovered  (se fue y regresó)
SINO SI gap_max >= 90d O días_sin_tx >= 90d:
    → churned    (inactivo prolongado)
SINO SI días_sin_tx <= 30d Y gap_max < 45d:
    → healthy    (activo sin interrupciones)
SINO:
    → churned    (caso borde → conservador)
```

**¿Por qué `recovered` tiene prioridad sobre `churned`?**
El cliente recovered SÍ tuvo un gap ≥ 45d (lo cual lo haría churned por la segunda regla),
pero como **regresó**, queremos diferenciarlo para aprender qué lo trajo de vuelta.

**Clientes sin ninguna transacción:** reciben `churned` automáticamente
(aparecen en `hey_clientes.csv` pero no en `hey_transacciones.csv`).

In [ ]:
# ── Celda 6: Construcción de etiquetas ───────────────────────────────────────

base = etiquetar_clientes(transacciones)

print('═══ PROCESO PASO A PASO ═══')
print(f'  Clientes en hey_clientes.csv         : {len(clientes):,}')
print(f'  Clientes con TXs (en base)           : {len(base):,}')
print(f'  Sin TXs (→ churned al mergear)       : {len(clientes) - len(base):,}')

print(f'\n─── Conteos intermedios ───')
print(f'  Con is_recovered = True              : {base["es_recovered"].sum():,}')
print(f'  Con gap_max >= {GAP_CHURNED}d                : {(base["max_gap_dias"] >= GAP_CHURNED).sum():,}')
print(f'  Activos (dias_sin_tx <= {VENTANA_ACTIVO_DIAS}d)        : {(base["dias_desde_ultima_tx"] <= VENTANA_ACTIVO_DIAS).sum():,}')

print(f'\n═══ DISTRIBUCIÓN FINAL ═══')
dist     = base['etiqueta'].value_counts()
dist_pct = (base['etiqueta'].value_counts(normalize=True) * 100).round(1)

for lbl in ['healthy', 'recovered', 'churned']:
    n_lbl = dist.get(lbl, 0)
    p_lbl = dist_pct.get(lbl, 0)
    bar   = '#' * int(p_lbl / 2)
    print(f'  {lbl:<12}: {n_lbl:>5,}  ({p_lbl:>5.1f}%)  {bar}')

# Estadísticas por grupo
print(f'\n─── Estadísticas temporales por grupo ───')
stats_grupo = base.groupby('etiqueta')[['max_gap_dias', 'dias_desde_ultima_tx']].mean().round(1)
display(stats_grupo.reindex(['healthy', 'recovered', 'churned']))

# Gráfica
orden = ['healthy', 'recovered', 'churned']
presentes = [l for l in orden if l in dist.index]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bars = axes[0].bar(
    presentes, [dist.get(l, 0) for l in presentes],
    color=[HEY_COLORS[l] for l in presentes], width=0.5, edgecolor='white'
)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{int(bar.get_height()):,}', ha='center', fontweight='bold')
axes[0].set_title('Distribución de etiquetas (n)', fontweight='bold')
axes[0].set_ylabel('Clientes')

axes[1].pie(
    [dist.get(l, 0) for l in presentes],
    labels=[f'{l}\n{dist_pct.get(l, 0):.1f}%' for l in presentes],
    colors=[HEY_COLORS[l] for l in presentes],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Proporción por grupo', fontweight='bold')

plt.suptitle('Resultado del etiquetado de churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Feature Engineering RFM

### ¿Por qué RFM? Recencia, Frecuencia y Monto como predictores de comportamiento

El modelo RFM (Recency-Frequency-Monetary) es el framework estándar en marketing bancario
para cuantificar el valor y riesgo de un cliente. Sus 8 features capturan dimensiones
complementarias del comportamiento transaccional:

| Feature | Dimensión RFM | Qué captura |
|---|---|---|
| `recencia` | R | Qué tan reciente fue la última interacción |
| `frecuencia` | F | Qué tan frecuente es el cliente |
| `monto_promedio` | M | Ticket promedio — proxy de valor económico |
| `monto_total` | M | LTV (Lifetime Value) acumulado |
| `max_gap_dias` | R | Peor episodio de inactividad histórica |
| `pct_transacciones_completadas` | F | Tasa de éxito — proxy de fricción en pagos |
| `pct_uso_app` | F | Engagement digital — menor = mayor riesgo de churn |
| `num_categorias_distintas` | F | Diversificación de uso — mayor = más maduro |

> Estos features se calculan con `build_rfm_features()` de `src/features.py`.
> Son complementarios a los features de perfil: capturan **comportamiento**, no características.

In [ ]:
# ── Celda 7: Feature engineering RFM ─────────────────────────────────────────

rfm = build_rfm_features(transacciones)

print(f'RFM calculado: {rfm.shape}  — una fila por cliente')
print(f'Clientes cubiertos: {rfm["user_id"].nunique():,}')

print('\n─── Primeras filas ───')
display(rfm.head())

RFM_FEATURES = [c for c in rfm.columns if c != 'user_id']

# Descripción de cada feature
FEATURE_INFO = {
    'recencia':                       ('R', 'Días desde la última TX hasta la fecha de corte. Menor = más activo.'),
    'frecuencia':                     ('F', 'Número total de transacciones históricas. Mayor = más enganchado.'),
    'monto_promedio':                 ('M', 'Ticket promedio por transacción en MXN. Proxy de valor por visita.'),
    'monto_total':                    ('M', 'Suma histórica de montos en MXN. LTV aproximado del cliente.'),
    'max_gap_dias':                   ('R', 'Peor gap histórico entre TXs consecutivas. Detecta episodios de abandono.'),
    'pct_transacciones_completadas':  ('F', '% de TXs con estatus completada. Mide fricción en el proceso de pago.'),
    'pct_uso_app':                    ('F', '% de TXs vía app móvil (ios/android/huawei). Digital engagement.'),
    'num_categorias_distintas':       ('F', 'Categorías MCC distintas utilizadas. Diversificación = mayor madurez.'),
}

print('\n═══ DESCRIPCIÓN DE FEATURES RFM ═══')
for feat, (dim, desc) in FEATURE_INFO.items():
    if feat in rfm.columns:
        print(f'  [{dim}] {feat:<40} {desc}')

# Histogramas: 2 filas × 4 cols
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, feat in enumerate(RFM_FEATURES):
    data   = rfm[feat].dropna()
    p95    = data.quantile(0.95)
    clipped = data.clip(0, p95)

    axes[i].hist(clipped, bins=40, color=HEY_GREEN, edgecolor='white', alpha=0.85)
    axes[i].axvline(data.mean(),   color=HEY_DARK,   lw=1.5, linestyle='--',
                    label=f'media={data.mean():.1f}')
    axes[i].axvline(data.median(), color=HEY_ORANGE, lw=1.5, linestyle='-',
                    label=f'med={data.median():.1f}')
    dim_label = FEATURE_INFO.get(feat, ('?', ''))[0]
    axes[i].set_title(f'[{dim_label}] {feat.replace("_", " ")}', fontweight='bold', fontsize=9)
    axes[i].legend(fontsize=7)

plt.suptitle('Distribuciones de features RFM (hasta p95 para claridad)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\n═══ ESTADÍSTICAS DESCRIPTIVAS — FEATURES RFM ═══')
display(rfm[RFM_FEATURES].describe().round(2))

---
## 8. Features de perfil del cliente

### Variables demográficas y de producto seleccionadas

**Criterios de inclusión:**
1. Correlación con `churn` > 0.05 en valor absoluto
2. < 30% de nulos (o imputables sin perder señal)
3. Sin riesgo de sesgo protegido (se excluye `sexo`)
4. No redundante con otra feature ya incluida

**Variables excluidas y por qué:**

| Variable | Motivo de exclusión |
|---|---|
| `sexo` | Variable protegida — excluirla evita sesgo discriminatorio en el modelo |
| `estado` / `ciudad` | 2.9% nulos + alta cardinalidad sin encodear |
| `nivel_educativo` | Proxy del ingreso ya cubierto por `ingreso_mensual_mxn` |
| `ocupacion` | Alta cardinalidad; AUC no mejora en pruebas previas |
| `canal_apertura` | Dato histórico de adquisición, no refleja comportamiento actual |
| `preferencia_canal` | Cubierto por `pct_uso_app` en features RFM |
| `recibe_remesas` | Correlación < 0.02 con churn |
| `idioma_preferido` | Sin variabilidad predictiva (>99% es_MX) |
| `patron_uso_atipico` | Dato de detección de fraude, no de churn |
| `usa_hey_shop` | Cubierto por `num_categorias_distintas` |
| `tiene_seguro` | Correlación marginal; no justifica añadir dimensionalidad |

In [ ]:
# ── Celda 8: Features de perfil del cliente ───────────────────────────────────

# Unir clientes con etiquetas para poder calcular correlación con churn
clientes_base = merge_clientes_etiquetados(clientes, base)
clientes_base['churn'] = (clientes_base['etiqueta'] == 'churned').astype(int)

# Features incluidas
INCLUIDAS   = [c for c in FEATURES_PERFIL if c in clientes_base.columns]
# Features disponibles pero excluidas
excluibles  = {'sexo', 'estado', 'ciudad', 'nivel_educativo', 'ocupacion',
               'canal_apertura', 'preferencia_canal', 'recibe_remesas',
               'idioma_preferido', 'patron_uso_atipico', 'usa_hey_shop',
               'tiene_seguro'}
EXCLUIDAS   = [c for c in clientes.columns
               if c in excluibles and c in clientes_base.columns]

# Correlación con churn
corr_df = clientes_base[INCLUIDAS + EXCLUIDAS + ['churn']].copy()
for col in corr_df.select_dtypes(include=['bool']).columns:
    corr_df[col] = corr_df[col].astype(int)
# Factorizar columnas object
for col in corr_df.select_dtypes(include=['object']).columns:
    corr_df[col] = pd.factorize(corr_df[col])[0]
corr_df = corr_df.fillna(corr_df.median(numeric_only=True))

corr_churn_all = corr_df.corr()['churn'].drop('churn').sort_values(key=abs, ascending=False)

print('═══ CORRELACIÓN CON CHURN ═══')
print('\nFeatures INCLUIDAS:')
for feat in INCLUIDAS:
    corr_val = corr_churn_all.get(feat, float('nan'))
    flag = ' ***' if abs(corr_val) > 0.15 else (' **' if abs(corr_val) > 0.08 else '')
    print(f'  {feat:<35}  corr={corr_val:+.3f}{flag}')

print('\nFeatures EXCLUIDAS (muestra):')
for feat in EXCLUIDAS[:6]:
    corr_val = corr_churn_all.get(feat, float('nan'))
    print(f'  {feat:<35}  corr={corr_val:+.3f}  (excluida)')

# Heatmap de correlaciones entre features incluidas + churn
corr_matrix = corr_df[INCLUIDAS + ['churn']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    cbar_kws={'shrink': 0.8, 'label': 'correlación de Pearson'}
)
ax.set_title('Heatmap de correlación — features de perfil + target churn',
             fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

# Barras de correlación con churn
corr_incluidas = corr_churn_all[INCLUIDAS].sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
colors = [HEY_DARK if v < 0 else HEY_GREEN for v in corr_incluidas.values]
ax.barh(corr_incluidas.index, corr_incluidas.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Correlación de features de perfil con churn', fontweight='bold')
ax.set_xlabel('Correlación de Pearson')
plt.tight_layout()
plt.show()

---
## 9. Dataset final para modelado

### El dataframe que entra al modelo

**Composición del dataset final:**
- **9 features de perfil** (demográficos + producto): capturan *quién es* el cliente
- **8 features RFM** (comportamiento transaccional): capturan *cómo se comporta*
- **Total: 17 features** + 1 target binario (`churn`)

**Imputaciones aplicadas por `build_model_features()`:**
- Clientes sin transacciones → RFM en 0 (excepto `recencia` = 999)
- `satisfaccion_1_10` nulos → se imputa con mediana **antes** de entrenar (en el modelo)
- Booleanos y categóricas → `int` o `pd.factorize()` antes de XGBoost

In [ ]:
# ── Celda 9: Dataset final para modelado ──────────────────────────────────────

# Preparar df_clientes con columna churn para build_model_features
df_cli_modelo = clientes_base[
    ['user_id', 'churn', 'etiqueta'] +
    [c for c in FEATURES_PERFIL if c in clientes_base.columns]
].copy()

df_model, FEATURE_COLS = build_model_features(df_cli_modelo, transacciones)

print(f'Shape del dataset final : {df_model.shape}')
print(f'Features en X           : {len(FEATURE_COLS)}')
print(f'Target: churn=1          {df_model["churn"].sum():,}  ({df_model["churn"].mean()*100:.1f}%)')
print(f'        churn=0          {(df_model["churn"]==0).sum():,}  ({(df_model["churn"]==0).mean()*100:.1f}%)')

# Verificar nulos en features
nulos_features = df_model[FEATURE_COLS].isnull().sum()
print(f'\n─── Nulos en features ───')
if nulos_features.sum() == 0:
    print('  Sin nulos — dataset listo para entrenar')
else:
    print('  ADVERTENCIA:')
    display(nulos_features[nulos_features > 0])

# Primeras filas
print('\n─── Primeras 5 filas ───')
display(df_model.head())

# Estadísticas del dataset final
print('\n═══ ESTADÍSTICAS DESCRIPTIVAS — DATASET FINAL ═══')
display(df_model[FEATURE_COLS].describe().round(2))

# Tabla de features con descripción
FEATURE_DESCRIPTIONS = {
    'score_buro':                      'Historial crediticio (295-850). Proxy de estabilidad financiera.',
    'satisfaccion_1_10':               'NPS autopercibido. El predictor anticipado más fuerte de churn.',
    'dias_desde_ultimo_login':         'Inactividad digital. Señal de desenganche antes del churn transaccional.',
    'num_productos_activos':           'Profundidad de relación con Hey Banco. Mayor = menor churn.',
    'ingreso_mensual_mxn':             'Capacidad económica. Segmentos bajos son más volátiles.',
    'es_hey_pro':                      'Subscripción premium. Clientes Pro tienen menor tasa de abandono.',
    'nomina_domiciliada':              'Dependencia del producto. Ancla al cliente a la cuenta.',
    'antiguedad_dias':                 'Madurez de la relación. Nuevos clientes son más proclives al churn.',
    'edad':                            'Correlación moderada. Segmentos 18-25 muestran mayor volatilidad.',
    'recencia':                        '[R] Días sin transaccionar. Señal directa de abandono en curso.',
    'frecuencia':                      '[F] Total de transacciones. Engagement histórico acumulado.',
    'monto_promedio':                  '[M] Ticket promedio en MXN. Proxy de valor por evento transaccional.',
    'monto_total':                     '[M] Lifetime value aproximado en MXN.',
    'max_gap_dias':                    '[R] Peor episodio de inactividad. Detecta patrones de abandono.',
    'pct_transacciones_completadas':   '[F] Tasa de TXs exitosas. Alta fricción correlaciona con churn.',
    'pct_uso_app':                     '[F] Porcentaje de uso de app móvil. Baja digitalización = mayor riesgo.',
    'num_categorias_distintas':        '[F] Diversidad de uso. Clientes maduros usan más categorías MCC.',
}

print('\n═══ FEATURES INCLUIDAS EN EL MODELO ═══')
print(f'{"Feature":<40} {"Descripción"}')
print('-' * 95)
for feat in FEATURE_COLS:
    desc = FEATURE_DESCRIPTIONS.get(feat, 'Sin descripción')
    print(f'{feat:<40} {desc}')

In [ ]:
# ── Celda 10: Guardar dataset final ──────────────────────────────────────────

Path(PROCESSED_PATH).mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_PATH + 'df_model_ready.csv'
df_model.to_csv(output_path, index=False, encoding='utf-8')

print(f'Dataset guardado en : {output_path}')
print(f'  Shape             : {df_model.shape[0]:,} filas × {df_model.shape[1]} columnas')
print(f'  Features          : {len(FEATURE_COLS)}')
print(f'  Target churn=1    : {df_model["churn"].sum():,}  ({df_model["churn"].mean()*100:.1f}%)')
print(f'  Target churn=0    : {(df_model["churn"]==0).sum():,}  ({(df_model["churn"]==0).mean()*100:.1f}%)')
print(f'  Ratio neg/pos     : {(df_model["churn"]==0).sum() / df_model["churn"].sum():.1f}x')

---
## 11. Resumen ejecutivo de decisiones

### Tabla de decisiones de limpieza

| # | Decisión | Afecta | Razonamiento |
|---|---|---|---|
| 1 | `fecha_hora` renombrado a `fecha` | Transacciones | Estandarización interna; el CSV usa nombre largo |
| 2 | Booleanos normalizados a `bool` | Clientes | Evita tratamiento como alta cardinalidad |
| 3 | `satisfaccion_1_10` nulos: **no imputar aquí** | Clientes | Se imputa en el modelo con mediana; preservar NaN revela que no todos dan NPS |
| 4 | `estado` / `ciudad` excluidas | Clientes | Nulos + cardinalidad sin encodear; geolocalización no mejora AUC |
| 5 | Outliers de `monto` e `ingreso`: **conservar** | Transacciones / Clientes | XGBoost es robusto; los valores extremos son reales y representan segmentos válidos |
| 6 | Umbral recovered: gap 45-90d con retorno | Etiquetado | Calibrado a frecuencia normal de uso y estándar CNBV de inactividad |
| 7 | Umbral churn: gap ≥ 90d o inactividad ≥ 90d | Etiquetado | Estándar bancario digital; 3 ciclos de nómina sin usar el producto |
| 8 | Clientes sin TXs → `churned` | Etiquetado | Nunca activaron la cuenta; son el churn más severo |

---

### Features finales y su justificación

**Features de perfil (9)** — capturan *quién es* el cliente:

| Feature | Por qué incluir |
|---|---|
| `satisfaccion_1_10` | Predictor más fuerte: insatisfacción precede al churn transaccional |
| `dias_desde_ultimo_login` | Desenganche digital = señal temprana antes del churn |
| `score_buro` | Estabilidad financiera; clientes con buen score son más estables |
| `num_productos_activos` | Profundidad de relación; cliente multiproducto tiene mayor costo de abandono |
| `ingreso_mensual_mxn` | Segmentos de bajo ingreso tienen mayor sensibilidad al churn |
| `es_hey_pro` | Suscripción con cobro mensual; abandono implica costo explícito |
| `nomina_domiciliada` | Ancla financiera; difícil de cambiar de banco si hay nómina domiciliada |
| `antiguedad_dias` | Los primeros 90 días son el período de mayor riesgo de abandono |
| `edad` | Diferencia comportamientos generacionales; jóvenes son más propensos a explorar alternativas |

**Features RFM (8)** — capturan *cómo se comporta* el cliente:

| Feature | Por qué incluir |
|---|---|
| `recencia` | El más predictivo del grupo RFM; largo silencio = alta probabilidad de churn |
| `max_gap_dias` | Detecta episodios pasados de inactividad que se pueden repetir |
| `frecuencia` | Engagement histórico; menos transacciones = menor dependencia |
| `pct_uso_app` | Desenganche digital antes de desenganche financiero |
| `monto_promedio` | Ticket bajo correlaciona con menor dependencia del producto |
| `monto_total` | LTV; mayor inversión acumulada = mayor motivación para quedarse |
| `num_categorias_distintas` | Madurez de uso; más categorías = cliente más integrado |
| `pct_transacciones_completadas` | Fricción acumulada en pagos puede motivar el abandono |

---

### Datos anómalos encontrados y cómo se manejaron

| Anomalía | Magnitud | Acción |
|---|---|---|
| Clientes sin ninguna transacción | Presente (ver celda 6) | Etiquetados como `churned`; RFM imputado en 0 |
| Montos muy altos (cola larga) | Algunos > $50,000 MXN | Conservados; XGBoost robusto, son segmento premium válido |
| NPS faltante (~5% clientes) | 751 nulos | Imputar mediana por grupo en el pipeline del modelo |
| Primer gap NaN por cliente | 802k - n_clientes registros | Esperado; la primera TX no tiene anterior — no imputar |
| Desbalance severo (churned ~2.4%) | 15:1 ratio neg/pos | Manejado con `scale_pos_weight` en XGBoost (notebook 03_modelo) |